# Google Meet + Vimeo QoE Report


In [ ]:
from pathlib import Path
import json
import os
import math
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import sys
REPO_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.git').exists())
sys.path.insert(0, str(REPO_ROOT / 'experiments/designated_experiments'))
from graph_output import install_graph_saver

BANDWIDTH_MBPS = int(os.environ.get('BANDWIDTH_MBPS', '6'))
result_candidates = [
    Path('meet_vimeo_youtube_experiment/meet_vimeo'),
    REPO_ROOT / 'experiments/meet_vimeo_youtube_experiment/meet_vimeo',
]
RESULT_DIR = next((p.resolve() for p in result_candidates if p.is_dir()), None)
if RESULT_DIR is None:
    raise FileNotFoundError('Could not locate meet_vimeo_youtube_experiment/meet_vimeo')

def load_qoe(app):
    path = RESULT_DIR / f'{app}_stats.jsonl'
    if not path.is_file():
        raise FileNotFoundError(f'Missing required QoE data: {path}')
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]

vimeo_qoe = load_qoe('vimeo')
meet_qoe = load_qoe('google_meet')
start_time = min(vimeo_qoe[0]['timestamp'], meet_qoe[0]['timestamp'])
vimeo_seconds = [row['timestamp'] - start_time for row in vimeo_qoe]
meet_seconds = [row['timestamp'] - start_time for row in meet_qoe]
plt.style.use('seaborn-v0_8-whitegrid')
GRAPHS_DIR = install_graph_saver(plt, RESULT_DIR, [
    '01_download_throughput_vimeo_meet',
    '02_upload_throughput_vimeo_meet',
    '03_buffer_health_vimeo_meet',
    '04_cumulative_dropped_frame_rate_vimeo_meet',
    '05_video_resolution_over_time_vimeo_meet',
    '06_total_video_frames_vimeo_meet',
])
VIMEO_COLOR, MEET_COLOR = '#1ab7ea', '#00897b'

vs = [row['stats'] for row in vimeo_qoe]
ms = [row['stats'] for row in meet_qoe]
summary = pd.DataFrame([{
    'Bandwidth': f'{BANDWIDTH_MBPS} Mbps',
    'Vimeo final resolution': vs[-1].get('resolution'),
    'Vimeo dropped frames': (vs[-1].get('dropped_video_frames', 0) or 0) - (vs[0].get('dropped_video_frames', 0) or 0),
    'Vimeo total frames': vs[-1].get('total_video_frames'),
    'Meet final resolution': ms[-1].get('resolution'),
    'Meet dropped frames': ms[-1].get('frames_dropped', 0) - ms[0].get('frames_dropped', 0),
    'Meet decoded frames': ms[-1].get('frames_decoded'),
    'Meet new freezes': ms[-1].get('freeze_count', 0) - ms[0].get('freeze_count', 0),
}]).set_index('Bandwidth')
display(summary.round(3))


## Per-application throughput

In [ ]:
PCAP = next(RESULT_DIR.glob('*.pcap'))
CLIENT_IP = '172.16.1.1'
window_start = min(vimeo_qoe[0]['timestamp'], meet_qoe[0]['timestamp'])
window_end = max(vimeo_qoe[-1]['timestamp'], meet_qoe[-1]['timestamp'])
window_seconds = window_end - window_start

sni_output = subprocess.run(
    ['tshark', '-r', str(PCAP), '-Y', 'tls.handshake.extensions_server_name',
     '-T', 'fields', '-e', 'ip.dst', '-e', 'tls.handshake.extensions_server_name'],
    check=True, capture_output=True, text=True,
).stdout
hosts_by_ip = {}
for line in sni_output.splitlines():
    fields = line.split('\t')
    if len(fields) >= 2 and fields[0] and fields[1]:
        hosts_by_ip.setdefault(fields[0], set()).update(h.lower() for h in fields[1].split(','))

vimeo_markers = ('vimeo.com', 'vimeocdn.com', 'akamaized.net')
meet_markers = ('meet.google.com', 'googleusercontent.com')
vimeo_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in vimeo_markers)}
meet_ips = {ip for ip, hosts in hosts_by_ip.items() if any(m in h for h in hosts for m in meet_markers)} - vimeo_ips

import ipaddress
google_networks = [ipaddress.ip_network(cidr) for cidr in (
    '172.217.0.0/16', '142.250.0.0/15', '142.251.0.0/16', '74.125.0.0/16',
    '64.233.160.0/19', '173.194.0.0/16', '108.177.0.0/17', '216.58.0.0/16',
    '34.104.0.0/16',
)]

def classify_ip(ip):
    if ip in vimeo_ips:
        return 'Vimeo'
    if ip in meet_ips:
        return 'Google Meet'
    addr = ipaddress.ip_address(ip)
    if any(addr in net for net in google_networks):
        return 'Google Meet'
    return 'Unclassified'

def classify_packets(direction_filter, remote_ip_field):
    output = subprocess.run(
        ['tshark', '-r', str(PCAP), '-Y', direction_filter, '-T', 'fields',
         '-e', 'frame.time_epoch', '-e', remote_ip_field, '-e', 'frame.len'],
        check=True, capture_output=True, text=True,
    ).stdout
    rows = []
    for line in output.splitlines():
        fields = line.split('\t')
        if len(fields) < 3 or not fields[0] or not fields[1] or not fields[2]:
            continue
        timestamp = float(fields[0])
        if not window_start <= timestamp <= window_end:
            continue
        remote_ip = fields[1].split(',')[0]
        frame_bytes = int(fields[2].split(',')[0])
        rows.append((timestamp, remote_ip, frame_bytes, classify_ip(remote_ip)))
    frame = pd.DataFrame(rows, columns=['timestamp', 'remote_ip', 'frame_bytes', 'application'])
    frame['second'] = (frame.timestamp - window_start).astype(int)
    return frame

download_packets = classify_packets(f'ip.dst == {CLIENT_IP}', 'ip.src')
upload_packets = classify_packets(f'ip.src == {CLIENT_IP}', 'ip.dst')

applications = ['Vimeo', 'Google Meet', 'Unclassified']
app_colors = {'Vimeo': VIMEO_COLOR, 'Google Meet': MEET_COLOR, 'Unclassified': '#999999'}
bin_count = max(1, math.ceil(window_seconds))

def per_second_mbps(packets):
    app_bytes = (packets[packets.application.isin(applications)]
                 .groupby(['second', 'application']).frame_bytes.sum()
                 .unstack(fill_value=0)
                 .reindex(range(bin_count), fill_value=0)
                 .reindex(columns=applications, fill_value=0))
    return app_bytes * 8 / 1_000_000

download_mbps = per_second_mbps(download_packets)
upload_mbps = per_second_mbps(upload_packets)

def summarize(packets, app_mbps, label):
    rows = []
    for app in applications:
        total_bytes = packets.loc[packets.application == app, 'frame_bytes'].sum()
        rows.append({
            'application': app,
            f'{label} (MiB)': total_bytes / 2**20,
            'average during QoE window (Mbps)': total_bytes * 8 / window_seconds / 1_000_000,
            'peak 1-second bin (Mbps)': app_mbps[app].max(),
        })
    display(pd.DataFrame(rows).set_index('application').round(3))

print('Download:')
summarize(download_packets, download_mbps, 'downloaded')
print('Upload:')
summarize(upload_packets, upload_mbps, 'uploaded')


## Download throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(download_mbps.index, download_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.axhline(BANDWIDTH_MBPS, color='#333333', linestyle='--', alpha=.7, label=f'Configured bottleneck ({BANDWIDTH_MBPS} Mbps)')
ax.set(title=f'Per-application Download Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Downloaded Mbit in each 1-second bin')
ax.set_ylim(0, 11)
ax.legend()
plt.tight_layout()
plt.show()


## Upload throughput

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
for app in applications:
    ax.plot(upload_mbps.index, upload_mbps[app], marker='o', color=app_colors[app], label=app, alpha=(1.0 if app != 'Unclassified' else 0.6))
ax.set(title=f'Per-application Upload Throughput — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Uploaded Mbit in each 1-second bin')
ax.legend()
plt.tight_layout()
plt.show()


## Buffer health

In [ ]:
vimeo_buffer = [row['stats'].get('buffer_ahead_secs', 0) or 0 for row in vimeo_qoe]
meet_jitter_buffer_ms = [
    1000 * (row['stats'].get('jitter_buffer_delay_seconds', 0) or 0)
    / max(1, row['stats'].get('jitter_buffer_emitted_count', 0) or 1)
    for row in meet_qoe
]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
meet_ax = ax.twinx()
buffer_line = ax.plot(vimeo_seconds, vimeo_buffer, marker='o', color=VIMEO_COLOR, label='Vimeo buffer ahead')[0]
jitter_line = meet_ax.plot(meet_seconds, meet_jitter_buffer_ms, marker='s', color=MEET_COLOR, label='Meet jitter buffer delay')[0]
ax.axhline(0, color='#ff7f7f', linestyle='--', alpha=.7)
ax.set(title=f'Buffer Health — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Vimeo buffer ahead (seconds)')
meet_ax.set_ylabel('Meet jitter buffer delay (ms, per emitted frame)')
ax.legend([buffer_line, jitter_line], ['Vimeo buffer ahead', 'Meet jitter buffer delay'], loc='best')
plt.tight_layout()
plt.show()


## Dropped frame rate

In [ ]:
vimeo_drop_pct = [
    100 * (row['stats'].get('dropped_video_frames', 0) or 0)
    / max(1, row['stats'].get('total_video_frames', 0) or 0)
    for row in vimeo_qoe
]
meet_drop_pct = [
    100 * (row['stats'].get('frames_dropped', 0) or 0)
    / max(1, (row['stats'].get('frames_decoded', 0) or 0) + (row['stats'].get('frames_dropped', 0) or 0))
    for row in meet_qoe
]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(vimeo_seconds, vimeo_drop_pct, marker='o', color=VIMEO_COLOR, label='Vimeo')
ax.plot(meet_seconds, meet_drop_pct, marker='o', color=MEET_COLOR, label='Google Meet')
ax.set(title=f'Cumulative Dropped Frame Rate — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Dropped frames (% of decoded frames so far)')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Video resolution over time

In [ ]:
vimeo_height = [row['stats'].get('video_height', 0) or 0 for row in vimeo_qoe]
vimeo_width = [row['stats'].get('video_width', 0) or 0 for row in vimeo_qoe]
meet_height = [row['stats'].get('frame_height', 0) or 0 for row in meet_qoe]
meet_width = [row['stats'].get('frame_width', 0) or 0 for row in meet_qoe]

fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.step(vimeo_seconds, vimeo_height, where='post', color=VIMEO_COLOR, linewidth=2, label='Vimeo')
ax.scatter(vimeo_seconds, vimeo_height, color=VIMEO_COLOR, s=24)
ax.step(meet_seconds, meet_height, where='post', color=MEET_COLOR, linewidth=2, label='Google Meet')
ax.scatter(meet_seconds, meet_height, color=MEET_COLOR, s=24)

resolution_labels = {}
for width, height in zip(vimeo_width + meet_width, vimeo_height + meet_height):
    if width and height:
        resolution_labels.setdefault(height, set()).add(f'{width}x{height}')
tick_heights = sorted(resolution_labels)
if tick_heights:
    ax.set_yticks(tick_heights, [' / '.join(sorted(resolution_labels[h])) for h in tick_heights])
ax.set(title=f'Video Resolution Over Time — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Rendered/decoded resolution')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## Total video frames

In [ ]:
vimeo_frames = [row['stats'].get('total_video_frames', 0) or 0 for row in vimeo_qoe]
meet_frames = [row['stats'].get('frames_decoded', 0) or 0 for row in meet_qoe]
fig, ax = plt.subplots(figsize=(10, 4.8), dpi=120)
ax.plot(vimeo_seconds, vimeo_frames, marker='o', color=VIMEO_COLOR, label='Vimeo')
ax.plot(meet_seconds, meet_frames, marker='o', color=MEET_COLOR, label='Google Meet')
ax.set(title=f'Total Video Frames — {BANDWIDTH_MBPS} Mbps', xlabel='Seconds from synchronized start', ylabel='Cumulative total video frames')
ax.legend(loc='best')
plt.tight_layout()
plt.show()
